## 🌟 Exercise 1 : Data Loading and Preparation

In this exercise, we will set up the environment and prepare the dataset that we will use throughout this project. Proper data preparation ensures smooth downstream processes, such as generating embeddings, working with vector databases, or building machine learning models.

### 1. Install Required Libraries

In [56]:
# On installe les librairies Python nécessaires
# Utilisation d'un 'or' pour tenter plusieurs versions de faiss-cpu si la première échoue
!pip install -q chromadb==1.5.9 sentence-transformers transformers requests pydantic-settings
!pip install -q faiss-cpu==1.8.0 || pip install -q faiss-cpu==1.7.4 || pip install -q faiss-cpu==1.7.2

In [57]:
# On crée un répertoire cache et on installe libomp-dev (dépendance système pour FAISS)
!mkdir -p cache
!apt install -y libomp-dev

# Re-upgrade faiss-cpu pour s'assurer qu'il est bien lié après l'installation de libomp-dev
!python -m pip install --upgrade faiss-cpu


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libomp-dev is already the newest version (1:14.0-55~exp2).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
  Using cached faiss_cpu-1.14.3-cp310-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (7.8 kB)
Using cached faiss_cpu-1.14.3-cp310-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (18.5 MB)
  Attempting uninstall: faiss-cpu
    Found existing installation: faiss-cpu 1.8.0
    Uninstalling faiss-cpu-1.8.0:
      Successfully uninstalled faiss-cpu-1.8.0


In [58]:
# On importe les librairies essentielles
import numpy as np
import pandas as pd
import faiss # Cette ligne devrait maintenant fonctionner si faiss est correctement installé
import json
import requests
from sentence_transformers import SentenceTransformer, InputExample
import chromadb
from chromadb.config import Settings
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline


### 3. Add an Identifier Column (if needed)

Unique identifiers help us track each record, especially when we work with vector databases.

In [59]:
# On charge le dataset dans un DataFrame pandas
# Télécharger le dataset de manière robuste
# **La tentative de téléchargement du dataset a échoué à plusieurs reprises avec une erreur 404.
# Pour ne pas bloquer l'exercice, nous allons créer un DataFrame de démonstration directement.**

import pandas as pd

# Création d'un DataFrame de démonstration
data = {
    'title': [
        "SpaceX launches new satellite to boost Starlink constellation",
        "NASA discovers new exoplanet with potential for life",
        "ESA plans ambitious Mars mission for 2028",
        "Breakthrough in AI research: new model achieves human-level text generation",
        "Latest report on climate change highlights urgent need for action",
        "Tech giants invest heavily in quantum computing startups",
        "Global economy shows signs of recovery despite inflation concerns",
        "New vaccine developed for emerging infectious disease",
        "Electric vehicles sales surge globally in Q3",
        "Researchers find ancient city ruins in Amazon rainforest"
    ],
    'topic': [
        "space", "space", "space", "technology", "environment",
        "technology", "economy", "health", "automotive", "archaeology"
    ]
}
pdf = pd.DataFrame(data)

# Ajout d'une colonne ID, nécessaire pour les exercices suivants
pdf["id"] = pdf.index.astype(str)

print("Dataset de démonstration créé et chargé dans un DataFrame.")
print("Premières lignes du DataFrame:")
display(pdf.head())

# Vous pouvez maintenant continuer avec les exercices en utilisant ce 'pdf' factice.

Dataset de démonstration créé et chargé dans un DataFrame.
Premières lignes du DataFrame:


,title,topic,id
0,SpaceX launches new satellite to boost Starlin...,space,0
1,NASA discovers new exoplanet with potential fo...,space,1
2,ESA plans ambitious Mars mission for 2028,space,2
3,Breakthrough in AI research: new model achieve...,technology,3
4,Latest report on climate change highlights urg...,environment,4


# On ajoute une colonne d'identifiant unique
pdf["id"] = pdf.index.astype(str)


In [60]:
# On ajoute une colonne d'identifiant unique
pdf["id"] = pdf.index.astype(str)


### 3. Add an Identifier Column (if needed)

Unique identifiers help us track each record, especially when we work with vector databases.

In [61]:
# On affiche les premières lignes du DataFrame et ses informations pour inspection
display(pdf.head())
display(pdf.info())

,title,topic,id
0,SpaceX launches new satellite to boost Starlin...,space,0
1,NASA discovers new exoplanet with potential fo...,space,1
2,ESA plans ambitious Mars mission for 2028,space,2
3,Breakthrough in AI research: new model achieve...,technology,3
4,Latest report on climate change highlights urg...,environment,4


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   title   10 non-null     object
 1   topic   10 non-null     object
 2   id      10 non-null     object
dtypes: object(3)
memory usage: 372.0+ bytes


None

### 5. Create a Subset for Faster Processing

To enable faster iterations during development, we'll select a smaller subset of the DataFrame (e.g., the first 1000 rows).

In [62]:
# Puisque nous utilisons un petit DataFrame de démonstration, nous n'avons pas besoin d'un sous-ensemble.
# Le DataFrame 'pdf' sera utilisé directement pour tous les exercices.
# Nous allons juste afficher les premières lignes du 'pdf' pour confirmer son état.
print(f"Shape of the full DataFrame: {pdf.shape}")
display(pdf.head())

Shape of the full DataFrame: (10, 3)


,title,topic,id
0,SpaceX launches new satellite to boost Starlin...,space,0
1,NASA discovers new exoplanet with potential fo...,space,1
2,ESA plans ambitious Mars mission for 2028,space,2
3,Breakthrough in AI research: new model achieve...,technology,3
4,Latest report on climate change highlights urg...,environment,4


## 🌟 Exercise 2: Vectorization with Sentence Transformers

In this exercise, we will transform our textual data (news titles) into numerical representations known as embeddings.

### 1. Install and Import Sentence Transformers Library

The `sentence_transformers` library provides easy-to-use methods for generating sentence-level embeddings.

In [63]:
# On importe la classe InputExample de sentence_transformers
from sentence_transformers import InputExample

### 2. Prepare the Data for Embedding Generation

We will apply a helper function to the subset of our DataFrame that we created earlier.

In [64]:
# On utilise le sous-ensemble du DataFrame créé précédemment
# pdf_subset est déjà défini à partir de l'exercice 1
# display(pdf_subset.head())

### 3. Create a Helper Function

This function converts each record (news title) into the proper format (`InputExample`) required by the Sentence Transformer model.

In [65]:
def example_create_fn(doc1: pd.Series) -> InputExample:
    """
    Helper function that outputs a sentence_transformer guid, label, and text.
    """
    # On crée un InputExample avec le titre de l'article comme texte
    # Le 'guid' peut être l'ID de l'article, et 'label' peut être le topic si disponible
    return InputExample(guid=doc1["id"], texts=[doc1["title"]], label=doc1["topic"])

### 4. Apply the Helper Function to the Subset

We’ll apply this function across the subset DataFrame to generate a list of `InputExample` objects:

In [66]:
# On applique la fonction d'aide à la colonne 'title' du DataFrame pour créer des InputExample
faiss_train_examples = pdf.apply(lambda x: example_create_fn(x), axis=1).tolist()
# On affiche les 10 premiers exemples pour vérification
print("Premiers 10 exemples:")
for i, example in enumerate(faiss_train_examples[:10]):
    print(f"  Exemple {i+1}: guid={example.guid}, texts={example.texts}, label={example.label})")

Premiers 10 exemples:
  Exemple 1: guid=0, texts=['SpaceX launches new satellite to boost Starlink constellation'], label=space)
  Exemple 2: guid=1, texts=['NASA discovers new exoplanet with potential for life'], label=space)
  Exemple 3: guid=2, texts=['ESA plans ambitious Mars mission for 2028'], label=space)
  Exemple 4: guid=3, texts=['Breakthrough in AI research: new model achieves human-level text generation'], label=technology)
  Exemple 5: guid=4, texts=['Latest report on climate change highlights urgent need for action'], label=environment)
  Exemple 6: guid=5, texts=['Tech giants invest heavily in quantum computing startups'], label=technology)
  Exemple 7: guid=6, texts=['Global economy shows signs of recovery despite inflation concerns'], label=economy)
  Exemple 8: guid=7, texts=['New vaccine developed for emerging infectious disease'], label=health)
  Exemple 9: guid=8, texts=['Electric vehicles sales surge globally in Q3'], label=automotive)
  Exemple 10: guid=9, texts=

### 5. Initialize the Embedding Model

We will use the pre-trained model `all-MiniLM-L6-v2`, which provides high-quality embeddings.

In [67]:
# On initialise le modèle SentenceTransformer pré-entraîné
# Assurez-vous que SentenceTransformer est importé (déjà fait au début de l'exercice 1)
model = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### 6. Extract the Titles and Convert to a List of Strings

Extract the “title” column from your DataFrame subset and convert it into a list.

In [51]:
# On extrait la colonne 'title' du DataFrame et on la convertit en liste
titles_list = pdf["title"].tolist()
print(f"Nombre de titres à embarquer: {len(titles_list)}")
print("Premiers 5 titres:", titles_list[:5])

Nombre de titres à embarquer: 10
Premiers 5 titres: ['SpaceX launches new satellite to boost Starlink constellation', 'NASA discovers new exoplanet with potential for life', 'ESA plans ambitious Mars mission for 2028', 'Breakthrough in AI research: new model achieves human-level text generation', 'Latest report on climate change highlights urgent need for action']


### 7. Generate Embeddings for the Titles

Using the initialized model, generate embeddings for each title:

In [68]:
# On génère les embeddings pour la liste des titres
faiss_title_embedding = model.encode(titles_list, show_progress_bar=True)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

### 8. Check Embedding Dimensions

To verify the embeddings were generated correctly, check the shape of the output:

In [69]:
# On vérifie la forme des embeddings générés
print(f"Nombre d'embeddings: {len(faiss_title_embedding)}")
print(f"Dimension de chaque embedding: {len(faiss_title_embedding[0])}")

Nombre d'embeddings: 10
Dimension de chaque embedding: 384


## 🌟 Exercise 3: FAISS Indexing and Search

In this exercise, we will use FAISS (Facebook AI Similarity Search) to build an index of the embeddings generated in the previous exercise.

### 1. Install and Import FAISS Library

Ensure FAISS is installed and import the necessary modules.

In [19]:
# numpy et faiss sont déjà importés dans la première cellule d'importation.
# import numpy as np
# import faiss

### 2. Prepare the Data for Indexing

Use the embedding vectors generated from the previous exercise and prepare them for indexing.

In [70]:
# On utilise le DataFrame et les embeddings générés
pdf_to_index = pdf.copy() # Copie pour éviter les SettingWithCopyWarning
# On convertit les IDs en tableau numpy d'entiers
id_index = np.array(pdf_to_index["id"].astype(int))

### 3. Normalize the Embedding Vectors

To perform cosine similarity search, we first need to normalize the embedding vectors.

In [71]:
# On normalise les vecteurs d'embeddings (L2 normalization)
content_encoded_normalized = faiss_title_embedding.astype('float32') # FAISS attend des float32
faiss.normalize_L2(content_encoded_normalized)
print("Vecteurs d'embeddings normalisés.")

Vecteurs d'embeddings normalisés.


### 4. Create the FAISS Index

We will use an `IndexFlatIP` (Inner Product) wrapped in an `IndexIDMap`.

In [72]:
# On crée l'index FAISS
# IndexFlatIP pour la similarité par produit interne (équivalent à la similarité cosinus pour vecteurs normalisés)
index_content = faiss.IndexIDMap(faiss.IndexFlatIP(len(faiss_title_embedding[0])))
# On ajoute les vecteurs normalisés avec leurs IDs à l'index
index_content.add_with_ids(content_encoded_normalized, id_index)
print(f"Index FAISS créé avec {index_content.ntotal} entrées.")

Index FAISS créé avec 10 entrées.


### 5. Implement a Search Function

Next, we’ll define a function `search_content` that takes a user query and retrieves the most similar articles from the index.

In [73]:
def search_content(query: str, pdf_to_index: pd.DataFrame, k: int = 3) -> pd.DataFrame:
    """
    Recherche les articles les plus similaires à une requête donnée dans l'index FAISS.
    """
    # On encode la chaîne de requête en un vecteur d'embedding
    query_vector = model.encode([query]).astype('float32')
    # On normalise le vecteur de requête
    faiss.normalize_L2(query_vector)

    # On effectue la recherche dans l'index FAISS
    similarities, ids = index_content.search(query_vector, k) # top-k vecteurs similaires et leurs IDs

    # On récupère les articles correspondants du DataFrame original
    # Les IDs retournés sont des ndarray, il faut les aplatir et les convertir en int
    results = pdf_to_index[pdf_to_index["id"].astype(int).isin(ids[0])].copy()
    results["similarities"] = similarities[0] # On ajoute les scores de similarité

    # Trie les résultats par similarité décroissante pour correspondre à l'ordre de FAISS
    results = results.set_index(pdf_to_index["id"].astype(int).loc[results.index])
    results = results.loc[ids[0]]
    results = results.reset_index(drop=True)

    return results

### 6. Test the Search Function

Use the search function to find articles related to a sample query.

In [74]:
# On teste la fonction de recherche avec un exemple de requête
query_term = "animal"
num_results = 5
print(f"Recherche des {num_results} articles les plus pertinents pour la requête: '{query_term}'")
search_results = search_content(query_term, pdf_to_index, k=num_results)
display(search_results)

Recherche des 5 articles les plus pertinents pour la requête: 'animal'


,title,topic,id,similarities
0,New vaccine developed for emerging infectious ...,health,7,0.046989
1,Breakthrough in AI research: new model achieve...,technology,3,0.085301
2,ESA plans ambitious Mars mission for 2028,space,2,0.094979
3,Latest report on climate change highlights urg...,environment,4,0.074539
4,Global economy shows signs of recovery despite...,economy,6,0.066922


## 🌟 Exercise 4: ChromaDB Collection and Querying

In this exercise, we will introduce ChromaDB, an open-source vector database designed to store, index, and query embedding vectors.

### 1. Install and Import ChromaDB Library

Ensure you have ChromaDB installed and import the necessary components.

In [75]:
# chromadb et Settings sont déjà importés dans la première cellule d'importation.
# import chromadb
# from chromadb.config import Settings

### 2. Initialize a ChromaDB Client and Create a Collection

ChromaDB organizes vectors into collections, which are similar to tables in a database.

In [76]:
# On initialise le client ChromaDB
chroma_client = chromadb.Client()
collection_name = "my_news"

# Si une collection avec le même nom existe, on la supprime pour éviter les conflits
# On doit lister toutes les collections et vérifier le nom
existing_collections = chroma_client.list_collections()
if any(coll.name == collection_name for coll in existing_collections):
    print(f"Suppression de la collection existante: '{collection_name}'")
    chroma_client.delete_collection(name=collection_name)

print(f"Création de la collection: '{collection_name}'")
collection = chroma_client.create_collection(name=collection_name)

Création de la collection: 'my_news'


### 3. Add Data to the Collection

ChromaDB simplifies data ingestion by automatically generating embeddings if you don’t supply a custom embedding model.

In [77]:
# On affiche le DataFrame (pour référence)
display(pdf.head())

# On ajoute les titres d'articles avec leurs topics et IDs à la collection ChromaDB
# Utilisation de tout le DataFrame 'pdf' qui est déjà petit.
collection.add(
    documents=pdf["title"].tolist(),
    metadatas=[{"topic": topic} for topic in pdf["topic"].tolist()],
    ids=[str(i) for i in pdf["id"].tolist()] # Les IDs doivent être des strings
)
print(f"Ajout de {collection.count()} documents à la collection '{collection_name}'.")

,title,topic,id
0,SpaceX launches new satellite to boost Starlin...,space,0
1,NASA discovers new exoplanet with potential fo...,space,1
2,ESA plans ambitious Mars mission for 2028,space,2
3,Breakthrough in AI research: new model achieve...,technology,3
4,Latest report on climate change highlights urg...,environment,4


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 28.0MiB/s]


Ajout de 10 documents à la collection 'my_news'.


### 4. Query the Collection

Finally, perform a search query to retrieve the most relevant documents based on a search term.

In [78]:
# On effectue une requête de recherche sur la collection ChromaDB
query_text = "space"
num_results = 10
results = collection.query(
    query_texts=[query_text], # La requête de recherche
    n_results=num_results # Le nombre de résultats à retourner
)

print(f"Résultats de la recherche pour la requête '{query_text}':")
print(json.dumps(results, indent=4))

Résultats de la recherche pour la requête 'space':
{
    "ids": [
        [
            "1",
            "3",
            "2",
            "5",
            "0",
            "9",
            "8",
            "6",
            "4",
            "7"
        ]
    ],
    "embeddings": null,
    "documents": [
        [
            "NASA discovers new exoplanet with potential for life",
            "Breakthrough in AI research: new model achieves human-level text generation",
            "ESA plans ambitious Mars mission for 2028",
            "Tech giants invest heavily in quantum computing startups",
            "SpaceX launches new satellite to boost Starlink constellation",
            "Researchers find ancient city ruins in Amazon rainforest",
            "Electric vehicles sales surge globally in Q3",
            "Global economy shows signs of recovery despite inflation concerns",
            "Latest report on climate change highlights urgent need for action",
            "New vaccine d

## 🌟 Exercise 5: Question Answering with Hugging Face Model

In this exercise, we will bring everything together by building a Question Answering (Q/A) system using a Hugging Face language model.

### 1. Install and Import the Transformers Library

The Hugging Face `transformers` library provides access to a variety of pre-trained language models.

In [29]:
# Les classes sont déjà importées dans la première cellule d'importation.
# from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

### 2. Initialize the Model and Tokenizer

Select a pre-trained model for text generation (e.g., GPT-2) and initialize both the model and its tokenizer.

In [79]:
# On spécifie l'ID du modèle Hugging Face à utiliser
model_id = "gpt2"

# On charge le tokenizer pour le modèle
tokenizer = AutoTokenizer.from_pretrained(model_id)

# On charge le modèle de langage causal
lm_model = AutoModelForCausalLM.from_pretrained(model_id)

print(f"Modèle '{model_id}' et tokenizer chargés.")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Modèle 'gpt2' et tokenizer chargés.


### 3. Create a Text Generation Pipeline

Set up a pipeline for text generation, which wraps the model and tokenizer into a convenient interface.

In [80]:
# On crée un pipeline de génération de texte
pipe = pipeline(
    "text-generation",
    model=lm_model,
    tokenizer=tokenizer,
    max_new_tokens=512,  # Nombre maximal de tokens à générer.
    device_map="auto",   # Utilise automatiquement les ressources GPU/CPU disponibles.
)

print("Pipeline de génération de texte créé.")

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Pipeline de génération de texte créé.


### 4. Construct a Prompt Template

The prompt includes both the retrieved context (from ChromaDB) and the user’s question. This way, the model generates a response informed by the relevant documents.

In [81]:
# On définit la question de l'utilisateur
question = "What's the latest news on space development?"

# On utilise les résultats de recherche de ChromaDB comme contexte
# On doit s'assurer que les 'results' de ChromaDB sont disponibles
# Si 'results' n'est pas encore défini (par exemple, si cette cellule est exécutée seule), on peut en créer un faux pour le test
if 'results' not in locals():
    results = {"documents": [["SpaceX launches new satellite.", "NASA discovers new planet.", "ESA plans Mars mission."]]}

# On concatène les documents récupérés pour former le contexte
# Utilisation des 3 premiers documents pour un contexte gérable
context = " ".join([f"#{doc}" for doc in results["documents"][0][:3]])

# On construit le template de prompt
prompt_template = f"Relevant context: {context}\n\n The user's question: {question}"

print("Prompt template construit:")
print(prompt_template)

Prompt template construit:
Relevant context: #NASA discovers new exoplanet with potential for life #Breakthrough in AI research: new model achieves human-level text generation #ESA plans ambitious Mars mission for 2028

 The user's question: What's the latest news on space development?


### 5. Generate a Response Using the Pipeline

Feed the prompt to the text generation pipeline and generate a response.

In [82]:
# On génère une réponse en utilisant le pipeline de génération de texte
lm_response = pipe(prompt_template)

# On affiche la réponse générée
print("Réponse du modèle de langage:")
print(lm_response[0]["generated_text"])

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Réponse du modèle de langage:
Relevant context: #NASA discovers new exoplanet with potential for life #Breakthrough in AI research: new model achieves human-level text generation #ESA plans ambitious Mars mission for 2028

 The user's question: What's the latest news on space development?


The user's answer: The space agency is working on a new space flight mission to Mars that will send a mission to the dwarf planet in 2028. The mission will be called the International Space Station's Mars Express and is expected to include a large asteroid, a rover and a lander.


The user's answer: The agency is working on a new space flight mission to Mars that will send a mission to the dwarf planet in 2028. The mission will be called the International Space Station's Mars Express and is expected to include a large asteroid, a rover and a lander. The user's answer: The space agency is working on a new space flight mission to Mars that will send a mission to the dwarf planet in 2028. The mission w

### 6. Experiment with Different Prompts and Context Windows

Try varying the question and the context size (e.g., using more or fewer retrieved documents) to observe how the model’s responses change.

In [83]:
# Exemple d'expérimentation avec une nouvelle question et un contexte différent
new_question = "Tell me about recent space discoveries."
# Supposons que nous ayons de nouveaux résultats de recherche ou que nous voulions utiliser plus de contexte
# Pour cet exemple, nous allons juste réutiliser les résultats existants

# On peut modifier la quantité de contexte en ajustant le slicing [0][:N]
context_for_experiment = " ".join([f"#{doc}" for doc in results["documents"][0][:2]]) # Utiliser seulement 2 documents

new_prompt_template = f"Relevant context: {context_for_experiment}\n\n The user's question: {new_question}"

print("Nouveau prompt template pour l'expérimentation:")
print(new_prompt_template)

# On génère une nouvelle réponse
new_lm_response = pipe(new_prompt_template)
print("Nouvelle réponse du modèle de langage pour l'expérimentation:")
print(new_lm_response[0]["generated_text"])

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Nouveau prompt template pour l'expérimentation:
Relevant context: #NASA discovers new exoplanet with potential for life #Breakthrough in AI research: new model achieves human-level text generation

 The user's question: Tell me about recent space discoveries.
Nouvelle réponse du modèle de langage pour l'expérimentation:
Relevant context: #NASA discovers new exoplanet with potential for life #Breakthrough in AI research: new model achieves human-level text generation

 The user's question: Tell me about recent space discoveries. Tell me about new data.

Answer: I've done both because I've been lucky enough to have been part of the team that first produced this picture on Hubble. The data is in this paper that shows how many new planets we've discovered and the number of exoplanets we've discovered.

The team at Hubble, led by Prof. Robert Böhlert of the University of Colorado at Boulder, took the first images of a new exoplanet (Jupiters) at the European Space Agency's (ESA) Eris Kepler